
# Module 3: Trident Configuration

## Exercise 1: Working with Trident

**Objectives**

This exercise focuses on enabling you to do the following:
  - Create a NAS back end
  - Create a storage class for a NAS back end
  - Provision a persistent volume claim with a NAS back end
  - Mount the volumes in a pod
  - Perform back-end management by using the tridentctl tool
  - Perform back-end management by using the kubectl tool
  - Configure customized naming conventions
  - Create a NAS economy back end
  - Provision NVMe namespaces using NetApp Trident

**Exercise Equipment**

In this exercise, you use the following systems.

| System                  | Host Name   | IP Addresses   | User Name (case sensitive) | Password  |
|-------------------------|-------------|----------------|----------------------------|-----------|
| Kubernetes Control Plane| kubmas1-1   | 192.168.0.61   | root                       | Netapp1!  |




**Prerequisites**

Before starting this exercise, you should take the following actions:

  - Set up your Integrated Development Environment (IDE)
  - Download the courseware GIT repository
  - Configure your IDE to have access to your Kubernetes clusters
  - Create svm0
  - Configure svm0 to use the NFS v3 protocol
  - Install Trident in your source Kubernetes cluster
  - Ensure iSCSI, NVMe and NFS are properly configured on your worker nodes in the source Kubernetes cluster


---
---

#### Task 1: Create a NAS back end

In this task, you create a back end that uses ONTAP NAS storage driver. 

You can create many other back ends. 

For more information, see https://docs.netapp.com/us-en/trident/trident-use/backends.html.

**NOTE**: 

When you unzip the trident tar file, the path trident-installer/sample-inputs provides many examples of configuration files.


---

To complete this task successfully, you must first complete Task 4 in the Module 1
exercise. 

In that task, you configure a storage VM (storage virtual machine, also known as SVM) for the NFS protocol.

---

If you are using terminal to execute kubectl commands, please change directories in your terminal to Exercise 3 folder.


In [1]:
pwd

/home/user/STRSW-ILT-UATWK-1/Exercise 3


---

Modify the [exercise3Task1.json](./exercise3Task1.json) file to add the appropriate settings;

Save the file as `exercise3Task1mod.json`:

  - Version: 1
  - Storage driver name: ontap-nas
  - Back-end name: c1-svm0-nfs-tbe
  - Management LIF: 192.168.0.30
  - Data LIF: 192.168.0.31
  - SVM: svm0
  - Username: vsadmin
  - Password: Netapp1!


<details> <summary>Solution  </summary>

[exercise3Task1.json](./Solutions/exercise3Task1.json)

```json
{
    "version": 1,
    "storageDriverName": "ontap-nas",
    "backendName": "c1-svm0-nfs-tbe",
    "managementLIF": "192.168.0.30",
    "dataLIF": "192.168.0.31",
    "svm": "svm0",
    "username": "vsadmin",
    "password": "Netapp1!"
}

---

The back-end definition is the only place that stores the credentials in plain text.

After you create the back end, usernames and passwords are encoded with Base64 and stored as Kubernetes secrets. 

Creating and updating a back end are the only operations that require knowledge of the credentials. 

These operations should be admin-only.



---

Verify that you are in the working directory where the tridentctl tool and the back-end JSON are present.


---


Create the back end by using the tridentctl tool:

`tridentctl -n trident create backend -f exercise3Task1mod.json`

Sample output:

```terminal
+---------------------+----------------+--------------------------------------+--------+------------+--------+
| NAME | STORAGE DRIVER | UUID | STATE | USER-STATE | VOLUMES|
+---------------------+----------------+--------------------------------------+--------+------------+--------+
| c1-svm0-nfs-tbe | ontap-nas | 804a70a5-5959-435d-a9c7-6357230f2e13 | online | normal | 0 |
+---------------------+----------------+--------------------------------------+--------+------------+--------+
```


---

In [2]:
tridentctl -n trident create backend -f exercise3Task1mod.json


+-----------------+----------------+--------------------------------------+--------+------------+---------+
|      NAME       | STORAGE DRIVER |                 UUID                 | STATE  | USER-STATE | VOLUMES |
+-----------------+----------------+--------------------------------------+--------+------------+---------+
| c1-svm0-nfs-tbe | ontap-nas      | efcf9ef9-c14b-446c-97d6-6389dbdf9173 | online | normal     |       0 |
+-----------------+----------------+--------------------------------------+--------+------------+---------+


---

Review the tridentctl logs:


In [3]:
tridentctl -n trident logs | tail -n 10


time="2025-07-09T08:48:16Z" level=debug msg="REST API call complete." Duration="972.61µs" Method=GET RequestURL=/trident/v1/backend/c1-svm0-nfs-tbe Route=GetBackend StatusCode=200 logLayer=rest_frontend requestID=70ef9861-a991-4e7d-8d72-121b176c65f6 requestSource=REST workflow="trident_rest=logger"
time="2025-07-09T08:48:26Z" level=debug msg="Logged EMS message." driver=ontap-nas logLayer=core requestID=831c1cd7-84fc-4186-a714-816a01cb8342 requestSource=REST workflow="backend=create"
time="2025-07-09T08:48:58Z" level=debug msg="REST API call received." Duration="24.684µs" Method=GET RequestURL=/trident/v1/version Route=GetVersion logLayer=rest_frontend requestID=f5d06808-d662-4837-b8f9-7b3ac9af9cc6 requestSource=REST workflow="trident_rest=logger"
time="2025-07-09T08:48:58Z" level=debug msg="REST API call complete." Duration="684.4µs" Method=GET RequestURL=/trident/v1/version Route=GetVersion StatusCode=200 logLayer=rest_frontend requestID=f5d06808-d662-4837-b8f9-7b3ac9af9cc6 requestSo

---
---

#### Task 2: Create a Storage Class for a NAS back end

In this task, you create a storage class that uses the NAS back end that you created in Task 1.


---

In your integrated development environment (IDE), open the [exercise3Task2.yaml](./exercise3Task2.yaml) file.


---

Add the correct `backendType` to the parameters.

This value is the `storageDriverName` from the back-end JSON.

<details> <summary> Solution </summary>
  
[exercise3Task2.yaml](./Solutions/exercise3Task2.yaml)

```yaml
apiVersion: storage.k8s.io/v1
kind: StorageClass
metadata:
  name: c1-svm0-nfs-sc
provisioner: csi.trident.netapp.io
parameters:
  backendType: ontap-nas
  storagePools: "c1-svm0-nfs-tbe:.*"

---

Add the correct storagePools:

` "c1-svm0-nfs-tbe.*"` 

and save the file `./exercise3Task2mod.yaml`.


---

Create the storage class:


In [4]:
kubectl create -f exercise3Task2mod.yaml


storageclass.storage.k8s.io/c1-svm0-nfs-sc created


---

Verify that the storage class is created:


In [5]:
kubectl get sc c1-svm0-nfs-sc


NAME             PROVISIONER             RECLAIMPOLICY   VOLUMEBINDINGMODE   ALLOWVOLUMEEXPANSION   AGE
c1-svm0-nfs-sc   csi.trident.netapp.io   Delete          Immediate           false                  7s


---

Review the storage class by using the tridentctl tool:


In [6]:
tridentctl -n trident get storageclass c1-svm0-nfs-sc -o json


{
  "items": [
    {
      "Config": {
        "version": "1",
        "name": "c1-svm0-nfs-sc",
        "attributes": {
          "backendType": "ontap-nas"
        },
        "storagePools": {
          "c1-svm0-nfs-tbe": [
            ".*"
          ]
        },
        "additionalStoragePools": null
      },
      "storage": {
        "c1-svm0-nfs-tbe": [
          "Cluster1_01_FC_1"
        ]
      }
    }
  ]
}


---
---

#### Task 3: Provision a Persistent Volume Claim with a NAS back end

In this task, you create a persistent volume claim (PVC) for a volume that uses the storage class that you created.


---

In your IDE, open the [exercise3Task3.yaml](./exercise3Task3.yaml) file.


---

Update the `storageClassName` with the name of the storage class that you created in the previous task,

and then save the file as ./exercise3Task3mod.yaml.

<details> <summary>Solution  </summary>  

[exercise3Task3.yaml](./Solutions/exercise3Task3.yaml)

```yaml
kind: PersistentVolumeClaim
apiVersion: v1
metadata:
  name: c1-svm0-nfs-pvc-1
  namespace: default
  annotations:
    trident.netapp.io/snapshotDirectory: "true"
spec:
  accessModes:
    - ReadWriteOnce
  resources:
    requests:
      storage: 1Gi
  storageClassName: c1-svm0-nfs-sc


---

Create the PVC for a pod to use later:


In [7]:
kubectl create -f exercise3Task3mod.yaml


persistentvolumeclaim/c1-svm0-nfs-pvc-1 created


---

After a moment, verify that you created the PVC:

Sample output:

```terminal
NAME STATUS VOLUME CAPACITY ACCESS MODES STORAGECLASS …
c1-svm0-nfs-pvc-1 Bound pvc-bf08cf3c-1a31… 1Gi RWO c1-svm0-nfs-sc …



In [8]:
kubectl -n default get pvc c1-svm0-nfs-pvc-1


NAME                STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS     VOLUMEATTRIBUTESCLASS   AGE
c1-svm0-nfs-pvc-1   Bound    pvc-6185e676-71ae-49c7-9d95-4d99db51638d   1Gi        RWO            c1-svm0-nfs-sc   <unset>                 30s


---

Navigate to ONTAP System Manager, and see the new volume that Trident created:

https://192.168.0.101/sysmgr/v4/storage/volumes.



In [9]:
ssh admin@cluster1 volume show -vserver svm0 



Last login time: 7/9/2025 06:48:15
Vserver   Volume       Aggregate    State      Type       Size  Available Used%
--------- ------------ ------------ ---------- ---- ---------- ---------- -----
svm0      nfs          Cluster1_01_FC_1 online RW          1GB    971.3MB    0%
svm0      svm0_root    Cluster1_01_FC_2 online RW         20MB    18.32MB    3%
svm0      trident_pvc_6185e676_71ae_49c7_9d95_4d99db51638d Cluster1_01_FC_1 online RW 1GB 1023MB  0%
3 entries were displayed.



In [10]:
ssh admin@cluster1 volume show -vserver svm0 -volume trident_pvc_6185e676_71ae_49c7_9d95_4d99db51638d



Last login time: 7/9/2025 08:59:07

                                      Vserver Name: svm0
                                       Volume Name: trident_pvc_6185e676_71ae_49c7_9d95_4d99db51638d
                                    Aggregate Name: Cluster1_01_FC_1
     List of Aggregates for FlexGroup Constituents: Cluster1_01_FC_1
                                   Encryption Type: none
                  List of Nodes Hosting the Volume: Cluster1-01
                                       Volume Size: 1GB
                                Volume Data Set ID: 1027
                         Volume Master Data Set ID: 2158073925
                                      Volume State: online
                                      Volume Style: flex
                             Extended Volume Style: flexvol
                           FlexCache Endpoint Type: none
                            Is Cluster-Mode Volume: true
                             Is Constituent Volume: false
                     N

---
---

#### Task 4: Mount the volumes in a pod

In this task, you create a NGINX pod that uses the persistent volume (PV) and creates a default webpage in the PV. 

This task includes a challenge step that asks you to expose the pod by using a NodePort service 
and then view your custom webpage.


In your IDE, review the [exercise3Task4-1.yaml](./exercise3Task4-1.yaml) file.



---

Set the claimName definition to the name of the PVC that you created in the previous task.

<details> <summary>  Solution</summary>  

[exercise3Task4-1.yaml](./Solutions/exercise3Task4-1.yaml)

```yaml
kind: Pod
apiVersion: v1
metadata:
  name: nfs-pod
  namespace: default
  labels:
    app: nfs-web
spec:
  volumes:
    - name: nfs-storage
      persistentVolumeClaim:
       claimName: c1-svm0-nfs-pvc-1
  containers:
    - name: nfs-container
      image: nginx:1.25-alpine-slim
      imagePullPolicy: IfNotPresent
      resources:
        requests:
          memory: "64Mi"
          cpu: "250m"
        limits:
          memory: "128Mi"
          cpu: "500m"
      ports:
        - containerPort: 80
          name: "http-server"
      volumeMounts:
        - mountPath: "/usr/share/nginx/html"
          name: nfs-storage

---

Save the file as exercise3Task4-1mod.yaml



---

Create the pod to use the Trident volume:


In [11]:
kubectl create -f exercise3Task4-1mod.yaml


pod/nfs-pod created


---

Verify that you created the pod:


In [12]:
kubectl -n default get pod nfs-pod


NAME      READY   STATUS              RESTARTS   AGE
nfs-pod   0/1     ContainerCreating   0          7s


In [13]:
kubectl -n default get pod nfs-pod


NAME      READY   STATUS    RESTARTS   AGE
nfs-pod   1/1     Running   0          18s


Connect to the pod:

View how the PVC is mounted in the container:

`# df -h`

In [14]:
kubectl -n default exec -it nfs-pod -- /bin/sh -c "df -h"


Filesystem                Size      Used Available Use% Mounted on
overlay                 306.4G      4.8G    286.1G   2% /
tmpfs                    64.0M         0     64.0M   0% /dev
/dev/mapper/ubuntu--vg-root
                        306.4G      4.8G    286.1G   2% /etc/hosts
/dev/mapper/ubuntu--vg-root
                        306.4G      4.8G    286.1G   2% /dev/termination-log
/dev/mapper/ubuntu--vg-root
                        306.4G      4.8G    286.1G   2% /etc/hostname
/dev/mapper/ubuntu--vg-root
                        306.4G      4.8G    286.1G   2% /etc/resolv.conf
shm                      64.0M         0     64.0M   0% /dev/shm
192.168.0.31:/trident_pvc_6185e676_71ae_49c7_9d95_4d99db51638d
                          1.0G    256.0K   1023.8M   0% /usr/share/nginx/html
tmpfs                   128.0M     12.0K    128.0M   0% /run/secrets/kubernetes.io/serviceaccount
tmpfs                     6.8G         0      6.8G   0% /proc/acpi
tmpfs                    64.0M         0    

In [15]:
kubectl -n default exec -it nfs-pod -- /bin/sh -c "df -h" |grep -iA1 trident


192.168.0.31:/trident_pvc_6185e676_71ae_49c7_9d95_4d99db51638d
                          1.0G    256.0K   1023.8M   0% /usr/share/nginx/html


---

Change the directory to the Trident persistent volume:

`# cd /usr/share/nginx/html`

Create an HTML file in the current directory:
`# echo '<html><body>Hello [your name] using NFS</body></html>' > index.html`



---

In [16]:
kubectl -n default exec -it nfs-pod -- /bin/sh -c "echo '<html><body>Hello nfs-pod using NFS</body></html>' >  /usr/share/nginx/html/index.html;"


In [17]:
kubectl -n default exec -it nfs-pod -- /bin/sh -c "cat  /usr/share/nginx/html/index.html"


<html><body>Hello nfs-pod using NFS</body></html>



---

Create a NodePort service and view the webpage contains your custom message:

[exercise3Task4-2.yaml](./exercise3Task4-2.yaml)


In [18]:

kubectl create -f exercise3Task4-2.yaml

service/nfs-web created


---

View the services:

`kubectl -n default get services`


Sample output:
```terminal
NAME TYPE CLUSTER-IP EXTERNAL-IP PORT(S) AGE
kubernetes ClusterIP 10.96.0.1 <none> 443/TCP 2d
nfs-web NodePort 10.106.85.27 <none> 80:31319/TCP 4m39s
```

In [19]:
kubectl -n default get services

NAME                 TYPE        CLUSTER-IP       EXTERNAL-IP   PORT(S)        AGE
kubernetes           ClusterIP   10.96.0.1        <none>        443/TCP        2d3h
manual-nfs-service   NodePort    10.103.147.197   <none>        80:30091/TCP   14h
nfs-web              NodePort    10.103.150.176   <none>        80:30429/TCP   10s


Open a browser to one of the Kubernetes cluster node IP addresses and the NodePort referenced in the previous step. 

For example: http://192.168.0.62:31319. 

You should see your index.html page displayed in the browser. NOTE: Use HTTP.


In [20]:
nodeport=$(kubectl describe service nfs-web|grep -oP 'NodePort:\s+<unset>\s+\K\d+')

echo "http://kubwor1-1:$nodeport"

http://kubwor1-1:30429


In [21]:
kubectl describe service nfs-web

Name:                     nfs-web
Namespace:                default
Labels:                   <none>
Annotations:              <none>
Selector:                 app=nfs-web
Type:                     NodePort
IP Family Policy:         SingleStack
IP Families:              IPv4
IP:                       10.103.150.176
IPs:                      10.103.150.176
Port:                     <unset>  80/TCP
TargetPort:               80/TCP
NodePort:                 <unset>  30429/TCP
Endpoints:                10.42.0.2:80
Session Affinity:         None
External Traffic Policy:  Cluster
Events:                   <none>


---
---

#### Task 5: Perform Back-End management by using the tridentctl tool

In this task, you investigate the tridentctl commands and delete the pod that hosted your custom webpage. 

You then re-create the pod and notice that the persistent volume, which reattached to the new pod and your custom webpage, has persisted.


---

jQuery is a json query tool which is installed on your jumphost:

it can be installed using

`sudo apt install -y jq`


---

Use the tridentctl tool to identify the storage class that is mapped to the correct backend 

(for your convenience, you can copy this command from the [exercise3Task5.txt](./exercise3Task5.txt) file):

tridentctl get backend -n trident -o json | jq '[.items[] | {backend: .name, storageClasses: [.storage[].storageClasses]|unique}]'


Sample output:
```json
[
  { "backend": "c1-svm0-nfs-tbe",
    "storageClasses": [
     [
       "c1-svm0-nfs-sc"
     ]
    ]   
  }
]


In [23]:
tridentctl get backend -n trident -o json 

{
  "items": [
    {
      "name": "c1-svm0-nfs-tbe",
      "backendUUID": "efcf9ef9-c14b-446c-97d6-6389dbdf9173",
      "protocol": "file",
      "config": {
        "aggregate": "",
        "autoExportCIDRs": [
          "0.0.0.0/0",
          "::/0"
        ],
        "autoExportPolicy": false,
        "aws": null,
        "backendName": "c1-svm0-nfs-tbe",
        "backendPools": [
          "eyJzdm1VVUlEIjoiMWZiZWYwMmQtNWJmMC0xMWYwLTgxMmQtMDA1MDU2ODhmZThlIiwiYWdncmVnYXRlIjoiQ2x1c3RlcjFfMDFfRkNfMSJ9"
        ],
        "chapInitiatorSecret": "\u003cREDACTED\u003e",
        "chapTargetInitiatorSecret": "\u003cREDACTED\u003e",
        "chapTargetUsername": "\u003cREDACTED\u003e",
        "chapUsername": "\u003cREDACTED\u003e",
        "clientCertificate": "",
        "clientPrivateKey": "\u003cREDACTED\u003e",
        "cloneSplitDelay": "10",
        "credentials": {
          "name": "\u003cREDACTED\u003e",
          "type": "\u003cREDACTED\u003e"
        },
        "dataLIF": "192.1

In [22]:
tridentctl get backend -n trident -o json | jq '[.items[] | {backend: .name, storageClasses: [.storage[].storageClasses]|unique}]'

[
  {
    "backend": "c1-svm0-nfs-tbe",
    "storageClasses": [
      [
        "c1-svm0-nfs-sc"
      ]
    ]
  }
]


In [25]:
tridentctl get backend -n trident -o json | jq '[.items[] | {backend: .name, storageClasses: [.storage[].storageClasses], DataLIF: [.config.dataLIF] |unique}]'

[
  {
    "backend": "c1-svm0-nfs-tbe",
    "storageClasses": [
      [
        "c1-svm0-nfs-sc"
      ]
    ],
    "DataLIF": [
      "192.168.0.31"
    ]
  }
]


---

You can delete and update the back end by using the tridentctl tool. 

For more information, 

see https://docs.netapp.com/us-en/trident/trident-use/backend_ops_tridentctl.html#create-a-backend.


---

Delete the NFS-supported pod:


In [26]:
kubectl -n default delete pod nfs-pod


pod "nfs-pod" deleted


---

Investigate the logs and see if the volume was deleted when the pod was deleted:


In [27]:
tridentctl logs -n trident | tail -n 20


time="2025-07-09T09:18:59Z" level=debug msg="Creating new storage pool list for backend." Backend=c1-svm0-nfs-tbe Method=ConstructExternal logLayer=core pool=Cluster1_01_FC_1 requestID=dc4693b0-7ef5-464f-86c3-68756efbcf8b requestSource=Kubernetes storageClass=c1-svm0-nfs-sc workflow="storage_class=update"
time="2025-07-09T09:18:59Z" level=debug msg="Node updated in cache." logLayer=csi_frontend name=kubmas1-1 requestID=24c0c050-089c-4742-ac66-09930b62ee50 requestSource=Kubernetes workflow="node=update"
time="2025-07-09T09:18:59Z" level=debug msg="Node updated in cache." logLayer=csi_frontend name=kubwor1-1 requestID=1f6b630c-82ea-49f3-850e-0f9af65a63c5 requestSource=Kubernetes workflow="node=update"
time="2025-07-09T09:18:59Z" level=debug msg="Node updated in cache." logLayer=csi_frontend name=kubwor1-2 requestID=bee8622e-008e-49be-bb26-95eed9a1aab0 requestSource=Kubernetes workflow="node=update"
time="2025-07-09T09:18:59Z" level=debug msg="Node updated in cache." logLayer=csi_frontend

---

Notice that the volume was just “unpublished” and answer the following questions: 

How would you delete the volume automatically when you delete the pod? 

Does the PVC still exist? 

Also  notice the finalizer that is associated with the PVC.


---

Navigate to ONTAP System Manager and notice that the volume that Trident created for the
NFS-based PVC is still there: 

https://192.168.0.101/sysmgr/v4/storage/volumes.


In [28]:
ssh admin@cluster1 volume show


Last login time: 7/9/2025 09:00:43
Vserver   Volume       Aggregate    State      Type       Size  Available Used%
--------- ------------ ------------ ---------- ---- ---------- ---------- -----
Cluster1-01 vol0       aggr0_Cluster1_01 online RW      1.58GB    310.9MB   79%
svm0      nfs          Cluster1_01_FC_1 online RW          1GB    971.2MB    0%
svm0      svm0_root    Cluster1_01_FC_2 online RW         20MB    18.29MB    3%
svm0      trident_pvc_6185e676_71ae_49c7_9d95_4d99db51638d Cluster1_01_FC_1 online RW 1GB 1023MB  0%
4 entries were displayed.



---

Re-create the pod to use the Trident NFS-provided volume:



In [29]:
kubectl create -f exercise3Task4-1mod.yaml


pod/nfs-pod created


---

Display the webpage and verify the webpage contains your custom message.


In [30]:
nodeport=$(kubectl describe service nfs-web|grep -oP 'NodePort:\s+<unset>\s+\K\d+')

echo "http://kubwor1-1:$nodeport"

http://kubwor1-1:30429


---

Do not destroy any objects. You use the objects in the next exercise.


---
---

#### Task 6: Perform Back-End management by using the kubectl tool

Previously, you created a back end by using the tridentctl tool. 

In this task, you create a back end by using the `TridentBackendConfig` custom resource (CR) with the credentials that are stored in a Kubernetes secret. 

For this task, you create a new SVM with the iSCSI protocol configured. 

NOTE:

SVMs allow multiple protocols. 

You could add the iSCSI configuration to svm0, but you create another SVM in this task to keep the two SVMs functionality separate. 

You also use the SVM administrator (vsadmin) and a separate management path for Trident to communicate with the SVMs.


---

Verify that you created the gateway-system namespace and that the operator pod is running in that namespace.


---

Review and execute the [exercise3Task6-1.yaml](./exercise3Task6-1.yaml) file to create an SVM called svm1 with the defined protocol in ONTAP Cluster 1. 

NOTE: 


Execute this file with a kubectl apply command, otherwise, an error will occur stating that the cluster1 admin’s secret is already created.


---

In [31]:
kubectl apply -f exercise3Task6-1.yaml

secret/ontap-cluster1-admin2 created
secret/ontap-svm1-admin created
storagevirtualmachine.gateway.netapp.com/svm1 created


CHALLENGE STEP: 

Review the logs of the manager container for the gateway-manager deployment’s pod to see the gateway operator in action:


`kubectl -n gateway-system logs gateway-operator-[unique id] -c manager`


In [32]:
go=$(kubectl -n gateway-system get pods|grep gateway-operator| awk '{print $1}');echo $go



gateway-operator-787cfc6cf4-slp4q


In [33]:
kubectl -n gateway-system logs $go -c manager

2025-07-08T11:36:00Z	INFO	setup	starting manager
2025-07-08T11:36:00Z	INFO	controller-runtime.metrics	Starting metrics server
2025-07-08T11:36:00Z	INFO	setup	disabling http/2
2025-07-08T11:36:00Z	INFO	starting server	{"name": "health probe", "addr": "[::]:8081"}
I0708 11:36:00.399534       1 leaderelection.go:257] attempting to acquire leader lease gateway-system/f2ac972d.gateway.netapp.com...
I0708 11:36:00.486819       1 leaderelection.go:271] successfully acquired lease gateway-system/f2ac972d.gateway.netapp.com
2025-07-08T11:36:00Z	DEBUG	events	gateway-operator-787cfc6cf4-slp4q_b633238b-df84-4ef6-ab56-24d01dbb2176 became leader	{"type": "Normal", "object": {"kind":"Lease","namespace":"gateway-system","name":"f2ac972d.gateway.netapp.com","uid":"8fb91d68-5441-4771-8ce2-f14abad0f649","apiVersion":"coordination.k8s.io/v1","resourceVersion":"170854"}, "reason": "LeaderElection"}
2025-07-08T11:36:00Z	INFO	Starting EventSource	{"controller": "storagevirtualmachine", "controllerGroup": "ga

---

Open a browser and go to https://192.168.0.101/

(which is your Cluster1 management LIF’s address).
 

---

For now, use the standard System Manager. 

Click the link: Not now. Sign in to System Manager. 

Skip this step if you already have selected System Manager instead of using NetApp BlueXP.


---

Authenticate with your ONTAP cluster by providing the following credentials:
 
 - Login as: admin
 
 - Password: Netapp1!


---


Click Sign In.


---

From the left pane, navigate to Storage > Storage VMs.


---

Review the settings of svm1 and verify that iSCSI and NVMe/TCP are configured.
Also verify that the Cluster1_01_FC_1 is an available local tier for this SVM.











In [34]:
ssh admin@cluster1  "vserver context svm1; vserver show -fields aggr-list;nvme show -instance;iscsi show -instance"


Last login time: 7/9/2025 09:22:17

Info: Use 'exit' command to return.

vserver aggr-list        
------- ---------------- 
svm1    Cluster1_01_FC_1 

  (vserver nvme show)

  Administrative Status: up
Discovery Subsystem NQN: nqn.1992-08.com.netapp:sn.d228835a5ca611f0812d00505688fe8e:discovery


             Target Name: iqn.1992-08.com.netapp:sn.d228835a5ca611f0812d00505688fe8e:vs.3
            Target Alias: svm1
   Administrative Status: up



In [1]:
ssh admin@cluster1  "vserver show -vserver svm1 -fields aggr-list;nvme show -vserver svm1  -instance;iscsi show -vserver svm1  -instance"


Last login time: 7/9/2025 09:27:44
vserver aggr-list        
------- ---------------- 
svm1    Cluster1_01_FC_1 

  (vserver nvme show)

           Vserver Name: svm1
  Administrative Status: up
Discovery Subsystem NQN: nqn.1992-08.com.netapp:sn.d228835a5ca611f0812d00505688fe8e:discovery


                 Vserver: svm1
             Target Name: iqn.1992-08.com.netapp:sn.d228835a5ca611f0812d00505688fe8e:vs.3
            Target Alias: svm1
   Administrative Status: up



---

Update the [exercise3Task6-2.yaml](./exercise3Task6-2.yaml) file with the details of the iSCSI SVM:
  
  - User name: vsadmin
  
  - Password: Netapp1!
  
  - Management LIF: 192.168.0.40
  
  - SVM: svm1

Save the file as `exercise3Task6-2mod.yaml`

NOTE: 

Generally, you should not specify a Data LIF for block protocols, otherwise, multipath 
would be disabled.

<details> <summary>  Solution</summary>  

[exercise3Task6-2.yaml](./Solutions/exercise3Task6-2.yaml)

---

Create the secret and the back end by using the kubectl tool:

In [2]:
kubectl create -f exercise3Task6-2mod.yaml

secret/c1-svm1-backend-secret created
tridentbackendconfig.trident.netapp.io/c1-svm1-iscsi-tbc created


---

In the Kubernetes IDE extension, ensure you are in the trident namespace. 

Navigate to **Clusters** > **source-admin@source** > **Custom Resources** >
**tridentbackendconfigs** > **c1-svm1-iscsi-tbc**.

This back end is the one that you created. 

The status should show the last operation status as success and the phase as bound.


---

Verify that you created the back-end configuration:


In [3]:
kubectl -n trident get tbc -o wide


NAME                BACKEND NAME        BACKEND UUID                           PHASE   STATUS    STORAGE DRIVER   DELETION POLICY
c1-svm1-iscsi-tbc   c1-svm1-iscsi-tbe   0185caf3-3397-4a8b-a6e1-369eb28c1379   Bound   Success   ontap-san        delete


---

Get details on the back-end configuration that you created:


In [4]:
kubectl -n trident describe tbc c1-svm1-iscsi-tbc


Name:         c1-svm1-iscsi-tbc
Namespace:    trident
Labels:       <none>
Annotations:  <none>
API Version:  trident.netapp.io/v1
Kind:         TridentBackendConfig
Metadata:
  Creation Timestamp:  2025-07-09T09:48:02Z
  Finalizers:
    trident.netapp.io
  Generation:        1
  Resource Version:  339388
  UID:               c828b274-2ac9-4eaf-865b-f7ccd6585026
Spec:
  Backend Name:  c1-svm1-iscsi-tbe
  Credentials:
    Name:               c1-svm1-backend-secret
  Management LIF:       192.168.0.40
  Storage Driver Name:  ontap-san
  Svm:                  svm1
  Version:              1
Status:
  Backend Info:
    Backend Name:         c1-svm1-iscsi-tbe
    Backend UUID:         0185caf3-3397-4a8b-a6e1-369eb28c1379
  Deletion Policy:        delete
  Last Operation Status:  Success
  Message:                Backend 'c1-svm1-iscsi-tbe' created
  Phase:                  Bound
Events:
  Type    Reason   Age   From                    Message
  ----    ------   ----  ----                    

---

Review and update the name of the back end in the YAML, and then create the storage class
in the [exercise3Task6-3.yaml](./exercise3Task6-3.yaml) file:

Save the file as **exercise3Task6-3mod.yaml**

<details> <summary>  Solution</summary>

[exercise3Task6-3.yaml](./Solutions/exercise3Task6-3.yaml)

```yaml
apiVersion: storage.k8s.io/v1
kind: StorageClass
metadata:
  name: c1-svm1-iscsi-sc
provisioner: csi.trident.netapp.io
parameters:
  backendType: ontap-san
  storagePools: "c1-svm1-iscsi-tbe:.*"

In [5]:
kubectl create -f exercise3Task6-3mod.yaml


storageclass.storage.k8s.io/c1-svm1-iscsi-sc created


---

Review and update the storage class name in the YAML, and then create the PVC in
[exercise3Task6-4.yaml](./exercise3Task6-4.yaml):


<details> <summary>  Solution</summary>

[exercise3Task6-4.yaml](./Solutions/exercise3Task6-4.yaml)  

```yaml
kind: PersistentVolumeClaim
apiVersion: v1
metadata:
  name: c1-svm1-iscsi-pvc-1
  namespace: default
spec:
  accessModes:
    - ReadWriteOnce
  resources:
    requests:
      storage: 1Gi
  storageClassName: c1-svm1-iscsi-sc

In [6]:
kubectl create -f exercise3Task6-4mod.yaml


persistentvolumeclaim/c1-svm1-iscsi-pvc-1 created


---

Navigate to ONTAP System Manager and see the new volume that Trident created:



https://192.168.0.101/sysmgr/v4/storage/volumes.


In [7]:
ssh admin@cluster1 "volume show -vserver svm1"


Last login time: 7/9/2025 09:43:58
Vserver   Volume       Aggregate    State      Type       Size  Available Used%
--------- ------------ ------------ ---------- ---- ---------- ---------- -----
svm1      svm1_root    Cluster1_01_FC_2 online RW         20MB    18.72MB    1%
svm1      trident_pvc_9a9eca58_fe3a_4991_be4a_2f3d483968fd Cluster1_01_FC_1 online RW 1.10GB 1.10GB  0%
2 entries were displayed.



---

Navigate to the LUNs in ONTAP System Manager and see the new LUN that Trident created:



https://192.168.0.101/sysmgr/v4/storage/luns.


In [8]:
ssh admin@cluster1 "lun show -vserver svm1"


Last login time: 7/9/2025 09:57:35
Vserver   Path                            State   Mapped   Type        Size
--------- ------------------------------- ------- -------- -------- --------
svm1      /vol/trident_pvc_9a9eca58_fe3a_4991_be4a_2f3d483968fd/lun0 online unmapped linux 1GB



---

Review and update the claim name in the YAML, and then create the pod in the
[exercise3Task6-5.yaml](./exercise3Task6-5.yaml) file:

Save the file as **exercise3Task6-5mod.yaml**


<details> <summary>  Solutions</summary>

[exercise3Task6-5.yaml](./Solutions/exercise3Task6-5.yaml)  

```yaml
kind: Pod
apiVersion: v1
metadata:
  name: san-pod
  namespace: default
  labels:
    app: san-web
spec:
  volumes:
    - name: san-storage
      persistentVolumeClaim:
       claimName: c1-svm1-iscsi-pvc-1
  containers:
    - name: san-container
      image: nginx:1.25-alpine-slim
      resources:
        requests:
          memory: "64Mi"
          cpu: "250m"
        limits:
          memory: "128Mi"
          cpu: "500m"
      ports:
        - containerPort: 80
          name: "http-server"
      volumeMounts:
        - mountPath: "/usr/share/nginx/html"
          name: san-storage

In [25]:
kubectl get pvc -n default

NAME                  STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS       VOLUMEATTRIBUTESCLASS   AGE
c1-svm0-nfs-pvc-1     Bound    pvc-6185e676-71ae-49c7-9d95-4d99db51638d   1Gi        RWO            c1-svm0-nfs-sc     <unset>                 3h16m
c1-svm1-iscsi-pvc-1   Bound    pvc-9a9eca58-fe3a-4991-be4a-2f3d483968fd   1Gi        RWO            c1-svm1-iscsi-sc   <unset>                 136m
manual-nfs-pvc        Bound    manual-nfs-pv                              1Gi        RWO,ROX,RWX                       <unset>                 18h


In [9]:
kubectl create -f exercise3Task6-5mod.yaml


pod/san-pod created


---

Verify that you created the pod:


In [10]:
kubectl -n default get pod san-pod


NAME      READY   STATUS              RESTARTS   AGE
san-pod   0/1     ContainerCreating   0          10s


In [11]:
kubectl -n default get pod san-pod


NAME      READY   STATUS    RESTARTS   AGE
san-pod   1/1     Running   0          21s


---

Connect to the pod:


`kubectl -n default exec -it san-pod -- /bin/sh`

---

View how the PVC is mounted in the container:

`# df -h`



---

Change the directory to the Trident persistent volume:

`# cd /usr/share/nginx/html`



---

Create a file in this location:

`# echo '<html><body>Hello [your name] using iSCSI</body></html>' > index.html`



---

Use Ctrl-D to exit the container’s shell.


In [12]:
kubectl -n default exec -it san-pod -- /bin/sh -c "df -h"


Filesystem                Size      Used Available Use% Mounted on
overlay                 306.4G      4.7G    286.1G   2% /
tmpfs                    64.0M         0     64.0M   0% /dev
/dev/mapper/ubuntu--vg-root
                        306.4G      4.7G    286.1G   2% /etc/hosts
/dev/mapper/ubuntu--vg-root
                        306.4G      4.7G    286.1G   2% /dev/termination-log
/dev/mapper/ubuntu--vg-root
                        306.4G      4.7G    286.1G   2% /etc/hostname
/dev/mapper/ubuntu--vg-root
                        306.4G      4.7G    286.1G   2% /etc/resolv.conf
shm                      64.0M         0     64.0M   0% /dev/shm
/dev/mapper/3600a09807770457a795d592f7666516a
                        973.4M     24.0K    906.2M   0% /usr/share/nginx/html
tmpfs                   128.0M     12.0K    128.0M   0% /run/secrets/kubernetes.io/serviceaccount
tmpfs                     6.8G         0      6.8G   0% /proc/acpi
tmpfs                    64.0M         0     64.0M   0% /proc

In [13]:
kubectl -n default exec -it san-pod -- /bin/sh -c "df -h" |grep -iA1 00a098


/dev/mapper/3600a09807770457a795d592f7666516a
                        973.4M     24.0K    906.2M   0% /usr/share/nginx/html


---

In [14]:

kubectl -n default exec -it san-pod -- /bin/sh -c "echo '<html><body>Hello san-pod using iSCSI</body></html>' >  /usr/share/nginx/html/index.html;"


---

CHALLENGE STEP: 

Create a NodePort service and view the custom webpage. 

Note:

Check the default namespace you selected in IDE Kubernetes Extension

An example of this can be found in Solutions/exercise3Task6-6.yaml.

<details> <summary>Solution  </summary> 

[exercise3Task6-6.yaml](./Solutions/exercise3Task6-6.yaml) 

```yaml
apiVersion: v1
kind: Service
metadata:
  name: san-web
  namespace: default
spec:
  type: NodePort
  ports:
    - port: 80
      targetPort: 80
  selector:
    app: san-web

In [16]:
kubectl apply -f exercise3Task6-6mod.yaml

service/san-web created


In [17]:
nodeport=$(kubectl -n default describe service san-web|grep -oP 'NodePort:\s+<unset>\s+\K\d+')

echo "http://kubwor1-1:$nodeport"

http://kubwor1-1:30547


---

CHALLENGE STEP: 

Delete the pod and verify that the LUN still exists.


In [18]:
kubectl -n default delete pod san-pod

pod "san-pod" deleted


In [19]:
ssh admin@cluster1 lun show -vserver svm1


Last login time: 7/9/2025 09:58:08
Vserver   Path                            State   Mapped   Type        Size
--------- ------------------------------- ------- -------- -------- --------
svm1      /vol/trident_pvc_9a9eca58_fe3a_4991_be4a_2f3d483968fd/lun0 online mapped linux 1GB



---

CHALLENGE STEP: 

Re-create the pod and view the webpage that contains your custom
message.


In [20]:
kubectl create -f exercise3Task6-5mod.yaml


pod/san-pod created


In [21]:
nodeport=$(kubectl -n default describe service san-web|grep -oP 'NodePort:\s+<unset>\s+\K\d+')

echo "http://kubwor1-1:$nodeport"

http://kubwor1-1:30547


In [22]:
ssh admin@cluster1 lun show -vserver svm1


Last login time: 7/9/2025 10:08:42
Vserver   Path                            State   Mapped   Type        Size
--------- ------------------------------- ------- -------- -------- --------
svm1      /vol/trident_pvc_9a9eca58_fe3a_4991_be4a_2f3d483968fd/lun0 online mapped linux 1GB



---

Do not destroy any objects. You use the objects in a later exercise.


---
---

#### Task 7: Configure customized naming conventions

In this task, you will work with a custom naming convention for a new `TridentBackendConfig` definition.

---

Within ONTAP System Manager, review the volumes names created by Trident by default.

Notice that default naming conventions for volumes looks something like this:

`trident_pvc_e018e7ab_a95b_4cb7_a366_85953d8fdec5`



---

Review the [exercise3Task7-1.yaml](./exercise3Task7-1.yaml) file.

Notice the following:

1. This single file creates a secret for cluster1’s admin credentials and then uses that secret in `TridentBackendConfig`.

2. In the `TridentBackendConfig` (tbc), the name of the tbc object is different than the backend. 
This is not necessary. We are just demonstrating that the tbc object and the backend can be different.

3. The storageclass’ storagepools is linked to the backend name designated in the tbc object.

---

Edit and save the [exercise3Task7-1.yaml](./exercise3Task7-1.yaml) file  as **exercise3Task7-1mod.yaml**  and replace the *change_me* in the following locations:

- nameTemplate: '{{ .labels.cluster }}_{{ .volume.Namespace }}_{{.volume.RequestName }}_{{ .config.BackendName }}'

- cluster: Cluster1

Note the following:

1. The labels.cluster in the name template will map to Cluster1 that we defined in the cluster label.

2. The volume namespace will be the persistent volume claim’s namespace while the volume RequestName will be name of the persistent volume claim.

3. Finally, the TridentBackendConfig’s BackendName will be appended to the end of the volume name.

<details> <summary>Solution  </summary>

[exercise3Task7-1.yaml](./Solutions/exercise3Task7-1.yaml)

```yaml

---

Create the secret, TridentBackendConfig and storageclass:


In [26]:

kubectl create -f exercise3Task7-1mod.yaml


secret/c1-svm0-backend-secret created
tridentbackendconfig.trident.netapp.io/c1-svm0-nfs-custom-tbc created
storageclass.storage.k8s.io/c1-svm0-nfs-custom-sc created


---


Next, create a pvc that uses the storageclass you created in this task by updating the *change_me* field in [exercise3Task7-2.yaml](./exercise3Task7-2.yaml). 

Save the files as **exercise3Task7-2mod.yaml** and then execute the file:


<details> <summary>Solution</summary>

[exercise3Task7-2.yaml](./Solutions/exercise3Task7-2.yaml)

```yaml

In [27]:
kubectl create -f exercise3Task7-2mod.yaml


persistentvolumeclaim/c1-svm0-nfs-custom-pvc-1 created


---

Review the PVC and PV created by exercise3Task7-2.yaml.


In [28]:
kubectl -n default get pvc  

NAME                       STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS            VOLUMEATTRIBUTESCLASS   AGE
c1-svm0-nfs-custom-pvc-1   Bound    pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca   1Gi        RWO            c1-svm0-nfs-custom-sc   <unset>                 18s
c1-svm0-nfs-pvc-1          Bound    pvc-6185e676-71ae-49c7-9d95-4d99db51638d   1Gi        RWO            c1-svm0-nfs-sc          <unset>                 3h32m
c1-svm1-iscsi-pvc-1        Bound    pvc-9a9eca58-fe3a-4991-be4a-2f3d483968fd   1Gi        RWO            c1-svm1-iscsi-sc        <unset>                 153m
manual-nfs-pvc             Bound    manual-nfs-pv                              1Gi        RWO,ROX,RWX                            <unset>                 18h


In [29]:
kubectl -n default get pv

NAME                                       CAPACITY   ACCESS MODES   RECLAIM POLICY   STATUS   CLAIM                              STORAGECLASS            VOLUMEATTRIBUTESCLASS   REASON   AGE
manual-nfs-pv                              1Gi        RWO,ROX,RWX    Recycle          Bound    default/manual-nfs-pvc                                     <unset>                          24h
pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca   1Gi        RWO            Delete           Bound    default/c1-svm0-nfs-custom-pvc-1   c1-svm0-nfs-custom-sc   <unset>                          27s
pvc-6185e676-71ae-49c7-9d95-4d99db51638d   1Gi        RWO            Delete           Bound    default/c1-svm0-nfs-pvc-1          c1-svm0-nfs-sc          <unset>                          3h33m
pvc-9a9eca58-fe3a-4991-be4a-2f3d483968fd   1Gi        RWO            Delete           Bound    default/c1-svm1-iscsi-pvc-1        c1-svm1-iscsi-sc        <unset>                          153m


In [30]:
kubectl apply -f exercise3Task7-2mod.yaml


persistentvolumeclaim/c1-svm0-nfs-custom-pvc-1 configured


In [32]:
kubectl delete pv pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca


persistentvolume "pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca" deleted



In [33]:
ssh admin@cluster1 volume show -vserver svm0


Last login time: 7/9/2025 12:38:50
Vserver   Volume       Aggregate    State      Type       Size  Available Used%
--------- ------------ ------------ ---------- ---- ---------- ---------- -----
svm0      Cluster1_default_c1_svm0_nfs_custom_pvc_1_c1_svm0_nfs_custom_tbe_0b475 Cluster1_01_FC_1 online RW 1GB 1023MB  0%
svm0      nfs          Cluster1_01_FC_1 online RW          1GB    971.0MB    0%
svm0      svm0_root    Cluster1_01_FC_2 online RW         20MB    18.12MB    4%
svm0      trident_pvc_6185e676_71ae_49c7_9d95_4d99db51638d Cluster1_01_FC_1 online RW 1GB 1023MB  0%
4 entries were displayed.



In [36]:
kubectl apply -f exercise3Task7-2mod.yaml


persistentvolumeclaim/c1-svm0-nfs-custom-pvc-1 unchanged


---

Review the PVC and PV created by exercise3Task7-2.yaml.


In [37]:
kubectl -n default get pvc  

NAME                       STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS            VOLUMEATTRIBUTESCLASS   AGE
c1-svm0-nfs-custom-pvc-1   Bound    pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca   1Gi        RWO            c1-svm0-nfs-custom-sc   <unset>                 15m
c1-svm0-nfs-pvc-1          Bound    pvc-6185e676-71ae-49c7-9d95-4d99db51638d   1Gi        RWO            c1-svm0-nfs-sc          <unset>                 3h48m
c1-svm1-iscsi-pvc-1        Bound    pvc-9a9eca58-fe3a-4991-be4a-2f3d483968fd   1Gi        RWO            c1-svm1-iscsi-sc        <unset>                 168m
manual-nfs-pvc             Bound    manual-nfs-pv                              1Gi        RWO,ROX,RWX                            <unset>                 18h


In [39]:
kubectl -n default get pv

NAME                                       CAPACITY   ACCESS MODES   RECLAIM POLICY   STATUS        CLAIM                              STORAGECLASS            VOLUMEATTRIBUTESCLASS   REASON   AGE
manual-nfs-pv                              1Gi        RWO,ROX,RWX    Recycle          Bound         default/manual-nfs-pvc                                     <unset>                          24h
pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca   1Gi        RWO            Delete           Terminating   default/c1-svm0-nfs-custom-pvc-1   c1-svm0-nfs-custom-sc   <unset>                          15m
pvc-6185e676-71ae-49c7-9d95-4d99db51638d   1Gi        RWO            Delete           Bound         default/c1-svm0-nfs-pvc-1          c1-svm0-nfs-sc          <unset>                          3h48m
pvc-9a9eca58-fe3a-4991-be4a-2f3d483968fd   1Gi        RWO            Delete           Bound         default/c1-svm1-iscsi-pvc-1        c1-svm1-iscsi-sc        <unset>                          168m


In [41]:
kubectl -n default describe pv pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca

Name:            pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca
Labels:          <none>
Annotations:     pv.kubernetes.io/provisioned-by: csi.trident.netapp.io
                 volume.kubernetes.io/provisioner-deletion-secret-name: 
                 volume.kubernetes.io/provisioner-deletion-secret-namespace: 
Finalizers:      [external-provisioner.volume.kubernetes.io/finalizer kubernetes.io/pv-protection]
StorageClass:    c1-svm0-nfs-custom-sc
Status:          Terminating (lasts 10m)
Claim:           default/c1-svm0-nfs-custom-pvc-1
Reclaim Policy:  Delete
Access Modes:    RWO
VolumeMode:      Filesystem
Capacity:        1Gi
Node Affinity:   <none>
Message:         
Source:
    Type:              CSI (a Container Storage Interface (CSI) volume source)
    Driver:            csi.trident.netapp.io
    FSType:            
    VolumeHandle:      pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca
    ReadOnly:          false
    VolumeAttributes:      backendUUID=76d2b164-86de-4fd2-9eb5-356435cc7472
     

In [42]:
kubectl -n default describe pv pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca

Name:            pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca
Labels:          <none>
Annotations:     pv.kubernetes.io/provisioned-by: csi.trident.netapp.io
                 volume.kubernetes.io/provisioner-deletion-secret-name: 
                 volume.kubernetes.io/provisioner-deletion-secret-namespace: 
Finalizers:      [external-provisioner.volume.kubernetes.io/finalizer kubernetes.io/pv-protection]
StorageClass:    c1-svm0-nfs-custom-sc
Status:          Terminating (lasts 29m)
Claim:           default/c1-svm0-nfs-custom-pvc-1
Reclaim Policy:  Delete
Access Modes:    RWO
VolumeMode:      Filesystem
Capacity:        1Gi
Node Affinity:   <none>
Message:         
Source:
    Type:              CSI (a Container Storage Interface (CSI) volume source)
    Driver:            csi.trident.netapp.io
    FSType:            
    VolumeHandle:      pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca
    ReadOnly:          false
    VolumeAttributes:      backendUUID=76d2b164-86de-4fd2-9eb5-356435cc7472
     

In [44]:
kubectl get  pvc 


NAME                       STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS            VOLUMEATTRIBUTESCLASS   AGE
c1-svm0-nfs-custom-pvc-1   Bound    pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca   1Gi        RWO            c1-svm0-nfs-custom-sc   <unset>                 38m
c1-svm0-nfs-pvc-1          Bound    pvc-6185e676-71ae-49c7-9d95-4d99db51638d   1Gi        RWO            c1-svm0-nfs-sc          <unset>                 4h10m
c1-svm1-iscsi-pvc-1        Bound    pvc-9a9eca58-fe3a-4991-be4a-2f3d483968fd   1Gi        RWO            c1-svm1-iscsi-sc        <unset>                 3h11m
manual-nfs-pvc             Bound    manual-nfs-pv                              1Gi        RWO,ROX,RWX                            <unset>                 18h


In [45]:
kubectl delete  pvc c1-svm0-nfs-custom-pvc-1


persistentvolumeclaim "c1-svm0-nfs-custom-pvc-1" deleted


In [46]:
kubectl get  pv


NAME                                       CAPACITY   ACCESS MODES   RECLAIM POLICY   STATUS   CLAIM                         STORAGECLASS       VOLUMEATTRIBUTESCLASS   REASON   AGE
manual-nfs-pv                              1Gi        RWO,ROX,RWX    Recycle          Bound    default/manual-nfs-pvc                           <unset>                          24h
pvc-6185e676-71ae-49c7-9d95-4d99db51638d   1Gi        RWO            Delete           Bound    default/c1-svm0-nfs-pvc-1     c1-svm0-nfs-sc     <unset>                          4h11m
pvc-9a9eca58-fe3a-4991-be4a-2f3d483968fd   1Gi        RWO            Delete           Bound    default/c1-svm1-iscsi-pvc-1   c1-svm1-iscsi-sc   <unset>                          3h12m


In [47]:
kubectl -n default describe pv pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca

Error from server (NotFound): persistentvolumes "pvc-0b47513a-f3a1-4834-bbb5-a933d366f5ca" not found


: 1

In [53]:
kubectl apply -f exercise3Task7-2mod.yaml


persistentvolumeclaim/c1-svm0-nfs-custom-pvc-1 created


In [54]:
kubectl get  pvc 


NAME                       STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS            VOLUMEATTRIBUTESCLASS   AGE
c1-svm0-nfs-custom-pvc-1   Bound    pvc-baa9f6bd-893e-41db-8df1-5dc89db46072   1Gi        RWO            c1-svm0-nfs-custom-sc   <unset>                 9s
c1-svm0-nfs-pvc-1          Bound    pvc-6185e676-71ae-49c7-9d95-4d99db51638d   1Gi        RWO            c1-svm0-nfs-sc          <unset>                 4h20m
c1-svm1-iscsi-pvc-1        Bound    pvc-9a9eca58-fe3a-4991-be4a-2f3d483968fd   1Gi        RWO            c1-svm1-iscsi-sc        <unset>                 3h20m
manual-nfs-pvc             Bound    manual-nfs-pv                              1Gi        RWO,ROX,RWX                            <unset>                 19h


In [55]:
kubectl get  pv

NAME                                       CAPACITY   ACCESS MODES   RECLAIM POLICY   STATUS   CLAIM                              STORAGECLASS            VOLUMEATTRIBUTESCLASS   REASON   AGE
manual-nfs-pv                              1Gi        RWO,ROX,RWX    Recycle          Bound    default/manual-nfs-pvc                                     <unset>                          25h
pvc-6185e676-71ae-49c7-9d95-4d99db51638d   1Gi        RWO            Delete           Bound    default/c1-svm0-nfs-pvc-1          c1-svm0-nfs-sc          <unset>                          4h20m
pvc-9a9eca58-fe3a-4991-be4a-2f3d483968fd   1Gi        RWO            Delete           Bound    default/c1-svm1-iscsi-pvc-1        c1-svm1-iscsi-sc        <unset>                          3h21m
pvc-baa9f6bd-893e-41db-8df1-5dc89db46072   1Gi        RWO            Delete           Bound    default/c1-svm0-nfs-custom-pvc-1   c1-svm0-nfs-custom-sc   <unset>                          21s


---

In [56]:
ssh admin@cluster1 volume show -vserver svm1


Last login time: 7/9/2025 12:43:26
Vserver   Volume       Aggregate    State      Type       Size  Available Used%
--------- ------------ ------------ ---------- ---- ---------- ---------- -----
svm1      svm1_root    Cluster1_01_FC_2 online RW         20MB    18.62MB    1%
svm1      trident_pvc_9a9eca58_fe3a_4991_be4a_2f3d483968fd Cluster1_01_FC_1 online RW 1.10GB 1.07GB  2%
2 entries were displayed.



In [57]:
kubectl describe pv pvc-baa9f6bd-893e-41db-8df1-5dc89db46072

Name:            pvc-baa9f6bd-893e-41db-8df1-5dc89db46072
Labels:          <none>
Annotations:     pv.kubernetes.io/provisioned-by: csi.trident.netapp.io
                 volume.kubernetes.io/provisioner-deletion-secret-name: 
                 volume.kubernetes.io/provisioner-deletion-secret-namespace: 
Finalizers:      [external-provisioner.volume.kubernetes.io/finalizer kubernetes.io/pv-protection]
StorageClass:    c1-svm0-nfs-custom-sc
Status:          Bound
Claim:           default/c1-svm0-nfs-custom-pvc-1
Reclaim Policy:  Delete
Access Modes:    RWO
VolumeMode:      Filesystem
Capacity:        1Gi
Node Affinity:   <none>
Message:         
Source:
    Type:              CSI (a Container Storage Interface (CSI) volume source)
    Driver:            csi.trident.netapp.io
    FSType:            
    VolumeHandle:      pvc-baa9f6bd-893e-41db-8df1-5dc89db46072
    ReadOnly:          false
    VolumeAttributes:      backendUUID=76d2b164-86de-4fd2-9eb5-356435cc7472
                       

Within ONTAP System Manager, discover the name of the volume created by
exercise3Task7-2.yaml.

NOTE: 

Trident automatically adds a suffix corresponding to a slice of the volume’s UUID (a part of the volume.Name).


In [58]:
ssh admin@cluster1 volume show -vserver svm0


Last login time: 7/9/2025 13:20:21
Vserver   Volume       Aggregate    State      Type       Size  Available Used%
--------- ------------ ------------ ---------- ---- ---------- ---------- -----
svm0      Cluster1_default_c1_svm0_nfs_custom_pvc_1_c1_svm0_nfs_custom_tbe_baa9f Cluster1_01_FC_1 online RW 1GB 1023MB  0%
svm0      nfs          Cluster1_01_FC_1 online RW          1GB    971.0MB    0%
svm0      svm0_root    Cluster1_01_FC_2 online RW         20MB    18.21MB    4%
svm0      trident_pvc_6185e676_71ae_49c7_9d95_4d99db51638d Cluster1_01_FC_1 online RW 1GB 1023MB  0%
4 entries were displayed.



In [59]:
kubectl get pvc

NAME                       STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS            VOLUMEATTRIBUTESCLASS   AGE
c1-svm0-nfs-custom-pvc-1   Bound    pvc-baa9f6bd-893e-41db-8df1-5dc89db46072   1Gi        RWO            c1-svm0-nfs-custom-sc   <unset>                 5m59s
c1-svm0-nfs-pvc-1          Bound    pvc-6185e676-71ae-49c7-9d95-4d99db51638d   1Gi        RWO            c1-svm0-nfs-sc          <unset>                 4h26m
c1-svm1-iscsi-pvc-1        Bound    pvc-9a9eca58-fe3a-4991-be4a-2f3d483968fd   1Gi        RWO            c1-svm1-iscsi-sc        <unset>                 3h26m
manual-nfs-pvc             Bound    manual-nfs-pv                              1Gi        RWO,ROX,RWX                            <unset>                 19h


In [60]:
kubectl delete pvc c1-svm0-nfs-custom-pvc-1

persistentvolumeclaim "c1-svm0-nfs-custom-pvc-1" deleted


In [61]:
kubectl get pvc

NAME                  STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS       VOLUMEATTRIBUTESCLASS   AGE
c1-svm0-nfs-pvc-1     Bound    pvc-6185e676-71ae-49c7-9d95-4d99db51638d   1Gi        RWO            c1-svm0-nfs-sc     <unset>                 4h28m
c1-svm1-iscsi-pvc-1   Bound    pvc-9a9eca58-fe3a-4991-be4a-2f3d483968fd   1Gi        RWO            c1-svm1-iscsi-sc   <unset>                 3h28m
manual-nfs-pvc        Bound    manual-nfs-pv                              1Gi        RWO,ROX,RWX                       <unset>                 19h


In [62]:
kubectl get pv pvc-baa9f6bd-893e-41db-8df1-5dc89db46072

Error from server (NotFound): persistentvolumes "pvc-baa9f6bd-893e-41db-8df1-5dc89db46072" not found


: 1

In [63]:
kubectl get sc c1-svm0-nfs-custom-sc

NAME                              PROVISIONER             RECLAIMPOLICY   VOLUMEBINDINGMODE   ALLOWVOLUMEEXPANSION   AGE
c1-svm0-nfs-custom-sc (default)   csi.trident.netapp.io   Delete          Immediate           false                  61m


---

In [64]:
kubectl get ns kube-system -o jsonpath='{.metadata.uid}'


ee54f070-5d49-45eb-8924-3b5b31a50cb7


CHALLENGE STEP: 

Experiment adding or replacing the name template in the TridentBackendConfig with other naming conventions such:

- {{ .config.StoragePrefix }}

- {{ slice .volume.Name }}

- Additional labels



---

CHALLENGE STEP: 

Discover the ONTAP volume name in the persistent volume’s internal
name attribute.


---

CHALLENGE STEP: 

Discover the default storagePrefix value if the TridentBackend (tbe) object
has {} as the value config.storage[0].ontap_config.storagePrefix.

Try:

`kubectl -n trident get tbe [some_tbe] -o jsonpath={".config.ontap_config.storage[0].defaults.nameTemplate"};echo`

In [65]:
kubectl -n trident get tbe

NAME        BACKEND                  BACKEND UUID
tbe-k6hf4   c1-svm1-iscsi-tbe        0185caf3-3397-4a8b-a6e1-369eb28c1379
tbe-mkmpb   c1-svm0-nfs-custom-tbe   76d2b164-86de-4fd2-9eb5-356435cc7472
tbe-tng2b   c1-svm0-nfs-tbe          efcf9ef9-c14b-446c-97d6-6389dbdf9173


In [68]:
kubectl -n trident get tbe tbe-mkmpb -o jsonpath={".config.ontap_config.storage[0].defaults.nameTemplate"};echo

{{.labels.cluster}}_{{.volume.Namespace}}_{{.volume.RequestName}}_{{.config.BackendName}}


In [69]:
kubectl -n default get pvc 

NAME                  STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS       VOLUMEATTRIBUTESCLASS   AGE
c1-svm0-nfs-pvc-1     Bound    pvc-6185e676-71ae-49c7-9d95-4d99db51638d   1Gi        RWO            c1-svm0-nfs-sc     <unset>                 4h47m
c1-svm1-iscsi-pvc-1   Bound    pvc-9a9eca58-fe3a-4991-be4a-2f3d483968fd   1Gi        RWO            c1-svm1-iscsi-sc   <unset>                 3h47m
manual-nfs-pvc        Bound    manual-nfs-pv                              1Gi        RWO,ROX,RWX                       <unset>                 19h


---
---

#### Task 8: Create a NAS economy back end

In this task, you will create a TridentBackendConfig to provisions qtrees instead of volumes, a storage class to use that backend configuration, and then a persistent volume claim to generate storage.



---

Review and update the *change_me* field in [exercise3Task8-1.yaml](./exercise3Task8-1.yaml).

Save the file as **exercise3Task8-1mod.yaml**


Create the secret, TridentBackendConfig and storageclass:

<details> <summary> Solution </summary> 

[exercise3Task8-1.yaml](./Solutions/exercise3Task8-1.yaml)
 
```yaml

In [70]:
kubectl apply -f exercise3Task8-1mod.yaml


secret/c1-svm0-backend-secret configured
tridentbackendconfig.trident.netapp.io/c1-svm0-nfs-eco-tbc created
storageclass.storage.k8s.io/c1-svm0-nfs-eco-sc created


---


Next, create a pvc that uses the storageclass you created in this task by updating the
*change_me* field in [exercise3Task8-2.yaml](./exercise3Task8-2.yaml).

Save the file as **exercise3Task8-2mod.yaml** and then execute the file:


<details> <summary>Solution </summary> 

[exercise3Task8-2.yaml](./Solutions/exercise3Task8-2.yaml)

 ```yaml

In [71]:
kubectl create -f exercise3Task8-2mod.yaml


persistentvolumeclaim/c1-svm0-nfs-eco-pvc-1 created


---

---

Review the PVC and PV created by exercise3Task8-2.yaml.


In [72]:
kubectl get pvc,pv

NAME                                          STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS         VOLUMEATTRIBUTESCLASS   AGE
persistentvolumeclaim/c1-svm0-nfs-eco-pvc-1   Bound    pvc-162fde14-0424-440b-86d6-82ec864d2122   1Gi        RWO            c1-svm0-nfs-eco-sc   <unset>                 37s
persistentvolumeclaim/c1-svm0-nfs-pvc-1       Bound    pvc-6185e676-71ae-49c7-9d95-4d99db51638d   1Gi        RWO            c1-svm0-nfs-sc       <unset>                 6h3m
persistentvolumeclaim/c1-svm1-iscsi-pvc-1     Bound    pvc-9a9eca58-fe3a-4991-be4a-2f3d483968fd   1Gi        RWO            c1-svm1-iscsi-sc     <unset>                 5h4m
persistentvolumeclaim/manual-nfs-pvc          Bound    manual-nfs-pv                              1Gi        RWO,ROX,RWX                         <unset>                 20h

NAME                                                        CAPACITY   ACCESS MODES   RECLAIM POLICY   STATUS   CLAIM               

In [73]:
kubectl get pvc c1-svm0-nfs-eco-pvc-1

NAME                    STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS         VOLUMEATTRIBUTESCLASS   AGE
c1-svm0-nfs-eco-pvc-1   Bound    pvc-162fde14-0424-440b-86d6-82ec864d2122   1Gi        RWO            c1-svm0-nfs-eco-sc   <unset>                 2m38s


---


Within ONTAP System Manager, locate the name of the volume created by exercise3Task8-2.yaml. 

Notice the naming structure of the volume and the qtrees.


In [74]:
ssh admin@cluster1 volume show -vserver svm0|grep eco

svm0      trident_qtree_pool_nas_eco_LYVBMVEONR Cluster1_01_FC_1 online RW 1GB 1023MB  0%


---

Create another persistent volume claim in [exercise3Task8-3.yaml](./exercise3Task8-3.yaml):

<details> <summary>Solution </summary> 

[exercise3Task8-3.yaml](./Solutions/exercise3Task8-3.yaml)

 ```yaml


In [77]:
kubectl create -f exercise3Task8-3mod.yaml


persistentvolumeclaim/c1-svm0-nfs-eco-pvc-2 created


---

Review the resulting qtree in ONTAP System Manager.


In [78]:
ssh admin@cluster1 qtree show -vserver svm0



Last login time: 7/9/2025 15:03:55
Vserver    Volume        Qtree        Style        Oplocks   Status
---------- ------------- ------------ ------------ --------- --------
svm0       nfs           ""           unix         enable    normal
svm0       svm0_root     ""           unix         enable    normal
svm0       trident_pvc_6185e676_71ae_49c7_9d95_4d99db51638d "" unix enable normal
svm0       trident_qtree_pool_nas_eco_LYVBMVEONR "" unix enable normal
svm0       trident_qtree_pool_nas_eco_LYVBMVEONR nas_eco_default_c1_svm0_nfs_eco_pvc_1_c1_svm0_nfs_eco_tbe_162fd unix enable normal
svm0       trident_qtree_pool_nas_eco_LYVBMVEONR nas_eco_default_c1_svm0_nfs_eco_pvc_2_c1_svm0_nfs_eco_tbe_e0f54 unix enable normal
6 entries were displayed.



#### Task 9: Provision NVMe namespaces using Trident

In this task, you create an NVMe back end and a storage class, and you use that storage class to
create a persistent volume for a pod. 

Previously, in Task 6, you configured svm1 to serve the NVMe/TCP protocol.


---

Update the [exercise3Task9-1.yaml](./exercise3Task9-1.yaml) file with the details of the NVMe functionality for svm1 by using the **ontap-san** storage driver:

Save the file as **exercise3Task9-1mod.yaml**

  - Management LIF: 192.168.0.40
  
  - SVM: svm1
  
  - sanType: nvme
  
  - useREST: true

NOTE:

You created the credentials secret, c1-svm1-backend-secret, previously.

<details> <summary>Solution </summary>

[exercise3Task9-1.yaml](./Solutions/exercise3Task9-1.yaml)  

```yaml

---


---

Create the secret and the back end by using the kubectl tool:


In [79]:
kubectl create -f exercise3Task9-1mod.yaml


tridentbackendconfig.trident.netapp.io/c1-svm1-nvme-tbc created


---

In the Kubernetes IDE extension, ensure you are in the trident namespace. 

Navigate to 

**Clusters** > 

  **source-admin@source** > 

**Custom Resources** >

**tridentbackendconfigs** > 

**c1-svm1-nvme-tbc**.

This back end is the one that you created. 

The status should show the last operation status as success and the phase as bound.


```yaml
apiVersion: trident.netapp.io/v1
kind: TridentBackendConfig
metadata:
  creationTimestamp: "2025-07-09T15:13:11Z"
  finalizers:
  - trident.netapp.io
  generation: 1
  name: c1-svm1-nvme-tbc
  namespace: trident
  resourceVersion: "380536"
  uid: c37dc50a-9ce6-411f-bf1d-2ab75d6fbecf
spec:
  backendName: c1-svm1-nvme-tbe
  credentials:
    name: c1-svm1-backend-secret
  managementLIF: 192.168.0.40
  sanType: nvme
  storageDriverName: ontap-san
  svm: svm1
  useREST: true
  version: 1
status:
  backendInfo:
    backendName: c1-svm1-nvme-tbe
    backendUUID: c749ada6-ed29-4526-abba-f58dffc6438e
  deletionPolicy: delete
  lastOperationStatus: Success
  message: Backend 'c1-svm1-nvme-tbe' created
  phase: Bound
```

---

Verify that you created the back-end configuration:



In [80]:
kubectl -n trident get tbc -o wide


NAME                     BACKEND NAME             BACKEND UUID                           PHASE   STATUS    STORAGE DRIVER      DELETION POLICY
c1-svm0-nfs-custom-tbc   c1-svm0-nfs-custom-tbe   76d2b164-86de-4fd2-9eb5-356435cc7472   Bound   Success   ontap-nas           delete
c1-svm0-nfs-eco-tbc      c1-svm0-nfs-eco-tbe      7bebace0-baf9-45a9-ab50-61bda2862462   Bound   Success   ontap-nas-economy   delete
c1-svm1-iscsi-tbc        c1-svm1-iscsi-tbe        0185caf3-3397-4a8b-a6e1-369eb28c1379   Bound   Success   ontap-san           delete
c1-svm1-nvme-tbc         c1-svm1-nvme-tbe         c749ada6-ed29-4526-abba-f58dffc6438e   Bound   Success   ontap-san           delete


---

Get details on the back-end configuration that you created:


In [81]:
kubectl -n trident describe tbc c1-svm1-nvme-tbc


Name:         c1-svm1-nvme-tbc
Namespace:    trident
Labels:       <none>
Annotations:  <none>
API Version:  trident.netapp.io/v1
Kind:         TridentBackendConfig
Metadata:
  Creation Timestamp:  2025-07-09T15:13:11Z
  Finalizers:
    trident.netapp.io
  Generation:        1
  Resource Version:  380536
  UID:               c37dc50a-9ce6-411f-bf1d-2ab75d6fbecf
Spec:
  Backend Name:  c1-svm1-nvme-tbe
  Credentials:
    Name:               c1-svm1-backend-secret
  Management LIF:       192.168.0.40
  San Type:             nvme
  Storage Driver Name:  ontap-san
  Svm:                  svm1
  Use REST:             true
  Version:              1
Status:
  Backend Info:
    Backend Name:         c1-svm1-nvme-tbe
    Backend UUID:         c749ada6-ed29-4526-abba-f58dffc6438e
  Deletion Policy:        delete
  Last Operation Status:  Success
  Message:                Backend 'c1-svm1-nvme-tbe' created
  Phase:                  Bound
Events:
  Type    Reason   Age    From                    Me

---

Review and update the name of the back end in the YAML, and then create the storage class in the exercise3Task9-2.yaml file:

Save the file as **exercise3Task9-2mod.yaml**

<details> <summary>Solution </summary> 

[exercise3Task9-2.yaml](./Solutions/exercise3Task9-2.yaml) 

```yaml

In [82]:

kubectl create -f exercise3Task9-2mod.yaml


storageclass.storage.k8s.io/c1-svm1-nvme-sc created


---

Review and update the storage class name in the YAML, and then create the PVC in the
[exercise3Task9-3.yaml](./exercise3Task9-3.yaml) file:

Save the file as exercise3Task9-3mod.yaml

<details> <summary>Solution </summary>

[exercise3Task9-3.yaml](./Solutions/exercise3Task9-3.yaml)  

```yaml

In [83]:
kubectl create -f exercise3Task9-3mod.yaml


persistentvolumeclaim/c1-svm1-nvme-pvc-1 created


---

Navigate to ONTAP System Manager and see the new volume that Trident created:

https://192.168.0.101/sysmgr/v4/storage/volumes.


In [84]:
ssh admin@cluster1 volume show -vserver svm1


Last login time: 7/9/2025 15:09:10
Vserver   Volume       Aggregate    State      Type       Size  Available Used%
--------- ------------ ------------ ---------- ---- ---------- ---------- -----
svm1      svm1_root    Cluster1_01_FC_2 online RW         20MB    18.57MB    2%
svm1      trident_pvc_82cc1a28_22d2_4b02_8287_c9f8afcdf99c Cluster1_01_FC_1 online RW 1.10GB 1.10GB  0%
svm1      trident_pvc_9a9eca58_fe3a_4991_be4a_2f3d483968fd Cluster1_01_FC_1 online RW 1.10GB 1.07GB  2%
3 entries were displayed.



---

Navigate to NVMe Namespaces in ONTAP System Manager and see the new namespace that
Trident created:

https://192.168.0.101/sysmgr/v4/storage/nvmeNamespaces.


In [85]:
ssh admin@cluster1 nvme namespace show -vserver svm1 


Last login time: 7/9/2025 15:24:02
  (vserver nvme namespace show)
Vserver Path                             State      Size Subsystem       NSID
------- -------------------------------- ------- ------- ---------- ---------
svm1
        /vol/trident_pvc_82cc1a28_22d2_4b02_8287_c9f8afcdf99c/namespace0 online 1GB - -



---

Review and update the claim name in the YAML, and then create the pod in the
[exercise3Task9-4.yaml](./exercise3Task9-4.yaml) file:

Save the file as exercise3Task9-4mod.yaml

<details> <summary>Solution </summary>  

[exercise3Task9-4.yaml](./Solutions/exercise3Task9-4.yaml)

```yaml

In [86]:

kubectl create -f exercise3Task9-4mod.yaml


pod/nvme-pod created


---

Verify that the pod was created successfully:


In [87]:

kubectl -n default get pod nvme-pod


NAME       READY   STATUS    RESTARTS   AGE
nvme-pod   1/1     Running   0          12s


---

CHALLENGE STEP: 

Expose the pod using a service and try to access it.



In [88]:
kubectl -n default expose pod nvme-pod --type=NodePort --name=nvme-service


service/nvme-service exposed


In [89]:
kubectl -n default exec -it nfs-pod -- /bin/sh -c "echo '<html><body>Hello nfs-pod using NFS</body></html>' >  /usr/share/nginx/html/index.html;"


In [90]:
kubectl -n default exec  -it nvme-pod -- /bin/sh -c "echo '<html><body>hello from nvme-pod </body></html>' > /usr/share/nginx/html/index.html"

In [91]:
nodeport=$(kubectl -n default describe service nvme-service|grep -oP 'NodePort:\s+<unset>\s+\K\d+')

echo "http://kubwor1-1:$nodeport"

http://kubwor1-1:32697


---
---

End of exercise